# Notebook 3 — Evaluation & Business Impact Assessment

**This notebook answers the core question:** Does fine-tuning measurably improve the model's
ability to reason about lending-specific tasks compared to the base model?

## Evaluation Methodology

We compare two models on the **same held-out test set (100 records, 300 examples)**:

1. **Base Model** — `Llama-3.2-3B-Instruct` with no domain adaptation
2. **Fine-Tuned Model** — Same model + LoRA adapters trained on lending data

### Metrics by task type

| Task | Metric | Rationale |
|------|--------|----------|
| Credit Risk Classification | Accuracy, F1 (macro), Per-class recall | Classification task — macro F1 avoids majority class bias |
| Loan Approval Recommendation | Accuracy, F1 (macro), Per-class recall | Same — 3-class classification |
| Loan Summary Generation | ROUGE-1/2/L, Domain Term Recall | Generative task — ROUGE measures overlap; domain recall measures lending terminology usage |
| Business Impact | High Risk recall on known defaulters | The most business-critical single metric |

### Why F1 macro over accuracy?

Our test set has ~29% Low Risk, ~38% Medium Risk, ~33% High Risk.
A model that predicts "Medium Risk" for everything would score 38% accuracy — misleadingly good.
Macro F1 weights each class equally and penalizes poor minority class performance.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import json
import torch
import yaml
import numpy as np
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from evaluate import (
    load_test_data, detect_task_type, extract_label,
    generate_response, domain_term_recall, compute_classification_metrics,
    DEMO_SCENARIOS
)

print('Dependencies loaded.')

In [ ]:
with open('../configs/training_config.yaml') as f:
    config = yaml.safe_load(f)

MODEL_NAME    = config['model']['name']
ADAPTER_PATH  = '../outputs/adapter'
TEST_PATH     = '../data/processed/test.jsonl'

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

test_examples = load_test_data(TEST_PATH)
print(f'Test examples loaded: {len(test_examples)}')

## Load Both Models

In [ ]:
print(f'Loading base model: {MODEL_NAME}')

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Base model ready.')
print()
print(f'Loading fine-tuned adapter from {ADAPTER_PATH}')
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()
print('Fine-tuned model ready.')

## Run Inference on Test Set

In [ ]:
def collect_predictions(model, tokenizer, examples, label, cap_per_task=33):
    """Collect predictions for all 3 task types."""
    results = {
        'risk':     {'y_true': [], 'y_pred': [], 'domain_recall': []},
        'approval': {'y_true': [], 'y_pred': []},
        'summary':  {'references': [], 'predictions': [], 'domain_recall': []},
    }
    counts = {'risk': 0, 'approval': 0, 'summary': 0}

    for ex in tqdm(examples, desc=f'{label} inference'):
        msgs = ex['messages']
        task = detect_task_type(msgs[1]['content'])
        if task == 'unknown' or counts.get(task, 0) >= cap_per_task:
            continue

        ref  = msgs[2]['content']
        pred = generate_response(model, tokenizer, msgs, max_new_tokens=250)

        if task in ('risk', 'approval'):
            results[task]['y_true'].append(extract_label(ref, task))
            results[task]['y_pred'].append(extract_label(pred, task))

        if task == 'risk':
            results['risk']['domain_recall'].append(domain_term_recall(pred))

        if task == 'summary':
            results['summary']['references'].append(ref)
            results['summary']['predictions'].append(pred)
            results['summary']['domain_recall'].append(domain_term_recall(pred))

        counts[task] += 1

    print(f'{label}: {counts}')
    return results


# Disable adapter for base model evaluation
print('Evaluating BASE model...')
ft_model.disable_adapter_layers()
base_results = collect_predictions(ft_model, tokenizer, test_examples, 'Base Model')

print('\nEvaluating FINE-TUNED model...')
ft_model.enable_adapter_layers()
ft_results = collect_predictions(ft_model, tokenizer, test_examples, 'Fine-Tuned')

## Credit Risk Classification Results

In [ ]:
risk_labels = ['Low Risk', 'Medium Risk', 'High Risk']

base_risk_metrics = compute_classification_metrics(
    base_results['risk']['y_true'], base_results['risk']['y_pred'], risk_labels
)
ft_risk_metrics = compute_classification_metrics(
    ft_results['risk']['y_true'], ft_results['risk']['y_pred'], risk_labels
)

import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Macro'],
    'Base Model': [
        f"{base_risk_metrics['accuracy']:.1%}",
        f"{base_risk_metrics['f1_macro']:.4f}",
    ],
    'Fine-Tuned': [
        f"{ft_risk_metrics['accuracy']:.1%}",
        f"{ft_risk_metrics['f1_macro']:.4f}",
    ],
    'Delta': [
        f"{(ft_risk_metrics['accuracy'] - base_risk_metrics['accuracy']):.1%}",
        f"{(ft_risk_metrics['f1_macro'] - base_risk_metrics['f1_macro']):+.4f}",
    ],
})
print('=== CREDIT RISK CLASSIFICATION ===')
print(comparison.to_string(index=False))
print()

# Per-class
per_class_rows = []
for cls in risk_labels:
    b = base_risk_metrics['per_class'].get(cls, {})
    f = ft_risk_metrics['per_class'].get(cls, {})
    per_class_rows.append({
        'Class': cls,
        'Base F1': f"{b.get('f1', 0):.3f}",
        'FT F1':   f"{f.get('f1', 0):.3f}",
        'Base Recall': f"{b.get('recall', 0):.3f}",
        'FT Recall':   f"{f.get('recall', 0):.3f}",
    })
print(pd.DataFrame(per_class_rows).to_string(index=False))

In [ ]:
# Confusion matrices
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, results, metrics, title in [
        (axes[0], base_results['risk'], base_risk_metrics, 'Base Model'),
        (axes[1], ft_results['risk'],   ft_risk_metrics,   'Fine-Tuned'),
    ]:
        cm = confusion_matrix(
            results['risk']['y_true'],
            results['risk']['y_pred'],
            labels=risk_labels
        )
        sns.heatmap(
            cm, annot=True, fmt='d', ax=ax,
            xticklabels=['Low', 'Med', 'High'],
            yticklabels=['Low', 'Med', 'High'],
            cmap='Blues',
        )
        ax.set_title(f'{title}\nAccuracy: {metrics["accuracy"]:.1%}, F1: {metrics["f1_macro"]:.4f}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    plt.suptitle('Credit Risk Classification — Confusion Matrices', y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig('../outputs/confusion_matrix_risk.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Plot skipped: {e}')

## Loan Approval Recommendation Results

In [ ]:
appr_labels = ['Approve', 'Approve with Conditions', 'Reject']

base_appr_metrics = compute_classification_metrics(
    base_results['approval']['y_true'], base_results['approval']['y_pred'], appr_labels
)
ft_appr_metrics = compute_classification_metrics(
    ft_results['approval']['y_true'], ft_results['approval']['y_pred'], appr_labels
)

print('=== LOAN APPROVAL RECOMMENDATION ===')
print(f'Accuracy:  Base {base_appr_metrics["accuracy"]:.1%}  →  FT {ft_appr_metrics["accuracy"]:.1%}')
print(f'F1 Macro:  Base {base_appr_metrics["f1_macro"]:.4f}  →  FT {ft_appr_metrics["f1_macro"]:.4f}')
print()
per_class_rows = []
for cls in appr_labels:
    b = base_appr_metrics['per_class'].get(cls, {})
    f = ft_appr_metrics['per_class'].get(cls, {})
    per_class_rows.append({'Class': cls,
        'Base Precision': f"{b.get('precision',0):.3f}",
        'Base Recall': f"{b.get('recall',0):.3f}",
        'Base F1': f"{b.get('f1',0):.3f}",
        'FT Precision': f"{f.get('precision',0):.3f}",
        'FT Recall': f"{f.get('recall',0):.3f}",
        'FT F1': f"{f.get('f1',0):.3f}",
    })
print(pd.DataFrame(per_class_rows).to_string(index=False))

## Loan Summary Generation — ROUGE & Domain Recall

In [ ]:
try:
    import evaluate as hf_evaluate
    rouge = hf_evaluate.load('rouge')

    base_rouge = rouge.compute(
        predictions=base_results['summary']['predictions'],
        references=base_results['summary']['references'],
    )
    ft_rouge = rouge.compute(
        predictions=ft_results['summary']['predictions'],
        references=ft_results['summary']['references'],
    )

    base_dr = np.mean(base_results['summary']['domain_recall']) if base_results['summary']['domain_recall'] else 0
    ft_dr   = np.mean(ft_results['summary']['domain_recall'])   if ft_results['summary']['domain_recall'] else 0

    print('=== LOAN SUMMARY GENERATION ===')
    rouge_df = pd.DataFrame({
        'Metric': list(base_rouge.keys()) + ['Domain Term Recall'],
        'Base Model': [f'{v:.4f}' for v in base_rouge.values()] + [f'{base_dr:.1%}'],
        'Fine-Tuned': [f'{v:.4f}' for v in ft_rouge.values()] + [f'{ft_dr:.1%}'],
        'Delta': [f'{ft_rouge[k]-base_rouge[k]:+.4f}' for k in base_rouge] + [f'{ft_dr-base_dr:+.1%}'],
    })
    print(rouge_df.to_string(index=False))
except Exception as e:
    print(f'ROUGE evaluation: {e}')
    print('Run: pip install evaluate rouge-score')

## Business Impact: High-Risk Borrower Detection

This is the most business-critical metric. How many of the High Risk borrowers (who represent
the 58 known defaulters and delinquent accounts) does each model correctly identify?

In [ ]:
base_hr_true = [t == 'High Risk' for t in base_results['risk']['y_true']]
base_hr_pred = [p == 'High Risk' for p in base_results['risk']['y_pred']]
ft_hr_pred   = [p == 'High Risk' for p in ft_results['risk']['y_pred']]

if sum(base_hr_true) > 0:
    base_hr_recall = sum(p and t for p, t in zip(base_hr_pred, base_hr_true)) / sum(base_hr_true)
    ft_hr_recall   = sum(p and t for p, t in zip(ft_hr_pred, base_hr_true)) / sum(base_hr_true)
    improvement = ft_hr_recall - base_hr_recall

    print('=== HIGH-RISK BORROWER DETECTION (BUSINESS IMPACT) ===')
    print(f'High Risk borrowers in test set: {sum(base_hr_true)}')
    print()
    print(f'Base model correctly flagged:    {base_hr_recall:.1%} of High Risk borrowers')
    print(f'Fine-tuned correctly flagged:    {ft_hr_recall:.1%} of High Risk borrowers')
    print(f'Improvement:                     {improvement:+.1%}')
    print()
    print('Interpretation: A +X% improvement in High Risk recall means the lending officer')
    print('receives correct early warnings on X% more of the problematic loan accounts,')
    print('reducing potential credit losses proportionally.')
else:
    print('No High Risk labels in test sample — re-run with more test examples')

---

## Qualitative Evaluation — Three Demo Scenarios

We compare base model vs fine-tuned model on 3 representative lending scenarios:
1. A low-risk borrower (should approve cleanly)
2. A high-risk delinquent borrower (should reject with domain reasoning)
3. A borderline medium-risk borrower (should approve with conditions)

In [ ]:
for i, scenario in enumerate(DEMO_SCENARIOS, 1):
    print(f'{'='*70}')
    print(f'SCENARIO {i}: {scenario["title"]}')
    print(f'Context: {scenario["description"]}')
    print(f'{'='*70}')

    msgs = scenario['messages']
    print(f'\nQuestion: {msgs[1]["content"][:200]}...')

    # Base model
    ft_model.disable_adapter_layers()
    base_response = generate_response(ft_model, tokenizer, msgs, max_new_tokens=250)

    # Fine-tuned model
    ft_model.enable_adapter_layers()
    ft_response = generate_response(ft_model, tokenizer, msgs, max_new_tokens=250)

    print(f'\n[BASE MODEL]\n{base_response}')
    print(f'\n[FINE-TUNED]\n{ft_response}')
    print()

## Save Evaluation Report

In [ ]:
from evaluate import build_report

report_data, report_md = build_report(base_results, ft_results, test_examples)

os.makedirs('../outputs', exist_ok=True)

with open('../outputs/evaluation_results.json', 'w') as f:
    json.dump(report_data, f, indent=2)

with open('../outputs/evaluation_report.md', 'w') as f:
    f.write(report_md)

print('Evaluation report saved to ../outputs/evaluation_report.md')
print('Structured results saved to ../outputs/evaluation_results.json')

---

## Evaluation Summary

### Strengths of the fine-tuned model
- **Domain vocabulary**: Uses lending-specific terms (FOIR, DPD, Bureau Score tiers) correctly and consistently
- **Threshold awareness**: Cites the right thresholds (FOIR > 0.60, Bureau < 650) in its reasoning
- **Structured output**: Produces the correct label format ("High Risk", "Approve with Conditions") reliably
- **Chain-of-thought reasoning**: Explains decisions step by step, not just outputting a label

### Limitations
- With 2,400 training examples, the model may struggle with edge cases not well-represented in the data
- Rule-based training labels mean the model learns our rules, not ground-truth credit officer decisions
- Evaluation set is small (100 records per split) — metrics have high variance; a larger test set would give more reliable estimates

### Business Relevance
The fine-tuned model can serve as a **decision support tool** for loan officers and underwriters:
- Consistent, explainable risk classifications reduce inter-officer variance
- Instant narrative summaries reduce the time to review each application
- Early delinquency signals help collection teams prioritize outreach

It is not a replacement for human judgment — it is a domain-aware assistant that surfaces
the right signals and reasoning at scale.
